# ABS Quarterly Industrial Disputes 6321

Working days lost to industrial disputes (cat. 6321.0.55.001).

## Python set-up

In [1]:
# system imports
from typing import Sequence

# analytic imports
import pandas as pd
import readabs as ra
from readabs import metacol as mc

# local imports
from abs_helper import get_abs_data
from mgplot import (
    line_plot_finalise,
    bar_plot_finalise,
    seastrend_plot_finalise,
    multi_start,
)

# pandas display settings
pd.options.display.max_rows = 999999
pd.options.display.max_columns = 999
pd.options.display.max_colwidth = 100
from decompose import decompose


In [2]:
# configuration constants
SHOW = False
plot_times = 0, -41  # full history, and last ~10 years (41 quarters)
LFOOTER = "Australia. Original series."


## Get data from ABS

In [3]:
abs_dict, meta, source, _ = get_abs_data("6321.0.55.001")


In [4]:
# list tables and latest available period
for name, table in abs_dict.items():
    print(f"{name}: {table.index[-1]}")

# latest quarter actually present in the data (used for breakdown labels)
LATEST = abs_dict["6321055001Table1"].index[-1]


6321055001Table1: 2026Q1
6321055001Table2a: 2026Q1
6321055001Table2b: 2026Q1
6321055001Table3a: 2026Q1
6321055001Table3b: 2026Q1
6321055001Table4a: 2025Q4
6321055001Table4b: 2025Q4
6321055001Table4c: 2025Q4


## Helper functions

In [5]:
def get_series(table: str, selectors: dict[str, str]) -> tuple[pd.Series, str]:
    """Return (series, units) selected robustly by description from a table."""
    _, sid, units = ra.find_abs_id(meta, {table: mc.table, **selectors})
    return abs_dict[table][sid].dropna(), units


def plot_wdl_line(
    series: pd.Series,
    units: str,
    title: str,
    lfooter: str = LFOOTER,
) -> None:
    """Recalibrate units and plot full-history + recent line charts."""
    series, units = ra.recalibrate(series, units)
    multi_start(
        series,
        function=line_plot_finalise,
        starts=plot_times,
        title=title,
        ylabel=units,
        width=2,
        rfooter=source,
        lfooter=lfooter,
        show=SHOW,
    )


def latest_year_breakdown(
    table: str,
    members: Sequence[str],
    did_template: str,
) -> tuple[pd.Series, str]:
    """Sum the latest four quarters of working days lost for each member."""
    values: dict[str, float] = {}
    units = ""
    for member in members:
        s, units = get_series(table, {did_template.format(member): mc.did})
        values[member] = s.tail(4).sum()
    return pd.Series(values), units


## Working days lost: Australia total

In [6]:
# quarterly working days lost, dispute total (Table 1)
wdl_q, units_q = get_series(
    "6321055001Table1",
    {"Working days Lost ;  Quarter ;  Dispute total ;": mc.did},
)
plot_wdl_line(wdl_q, units_q, "Industrial Disputes: Working Days Lost per Quarter")


## Seasonal decomposition: quarterly national working days lost

In [7]:
# multiplicative decomposition of the quarterly national series.
# COVID years are excluded from the seasonal estimate (disputes slumped 2020-21).
wdl_recal, dec_units = ra.recalibrate(wdl_q, units_q)
decomposed = decompose(
    wdl_recal,
    model="multiplicative",
    ignore_years=(2020, 2021),
)
multi_start(
    decomposed[["Seasonally Adjusted", "Trend"]],
    function=seastrend_plot_finalise,
    starts=plot_times,
    title="Industrial Disputes: Working Days Lost, Seasonal Decomposition",
    ylabel=dec_units,
    rfooter=source,
    lfooter="Australia. Seasonally adjusted using in-house methods.",
    show=SHOW,
)


In [8]:
# rolling annual: 12 months ended (Table 1) - smooths the spiky quarterly series
wdl_annual, units = get_series(
    "6321055001Table1",
    {"Working days Lost ;  12 months ended ;  Dispute total ;": mc.did},
)
wdl_annual, units = ra.recalibrate(wdl_annual, units)
line_plot_finalise(
    wdl_annual,
    title="Industrial Disputes: Working Days Lost (12 Months Ended)",
    ylabel=units,
    width=2,
    color="darkred",
    rfooter=source,
    lfooter=LFOOTER,
    show=SHOW,
)


## Working days lost per 1,000 employees

In [9]:
# workforce-normalised measure, all industries (Table 2b)
wdl_per1000, units = get_series(
    "6321055001Table2b",
    {"Working days lost per 1000 employees ;  All industries ;": mc.did},
)
plot_wdl_line(
    wdl_per1000,
    units,
    "Industrial Disputes: Working Days Lost per 1,000 Employees",
)


## Working days lost by industry (latest four quarters)

In [10]:
industries = [
    "Coal mining",
    "Other mining",
    "Metal product etc manufacturing",
    "Other manufacturing",
    "Construction",
    "Transport, postal & warehousing",
    "Education & training; health care & social assistance",
    "Other industries",
]
by_industry, units = latest_year_breakdown(
    "6321055001Table2a", industries, "Working days Lost ;  {} ;"
)
by_industry = by_industry.sort_values()
by_industry = by_industry.rename(
    index={
        "Education & training; health care & social assistance": "Education & training;\nhealth care & social assistance"
    }
)
by_industry, units = ra.recalibrate(by_industry, units)
bar_plot_finalise(
    by_industry,
    title=f"Working Days Lost by Industry: Year to {LATEST}",
    xlabel=units,
    horizontal=True,
    rfooter=source,
    lfooter=LFOOTER,
    show=SHOW,
)


## Working days lost by state (latest four quarters)

In [11]:
states = [
    "New South Wales",
    "Victoria",
    "Queensland",
    "South Australia",
    "Western Australia",
    "Tasmania",
    "Northern Territory",
    "Australian Capital Territory",
]
by_state, units = latest_year_breakdown(
    "6321055001Table3a", states, "Working days Lost ;  {} ;"
)
by_state = by_state.sort_values()
by_state, units = ra.recalibrate(by_state, units)
bar_plot_finalise(
    by_state,
    title=f"Working Days Lost by State: Year to {LATEST}",
    xlabel=units,
    horizontal=True,
    rfooter=source,
    lfooter=LFOOTER,
    show=SHOW,
)


## Watermark

In [12]:
%load_ext watermark
%watermark -u -n -t -v -iv


Last updated: Wed, 10 Jun 2026 17:30:21

Python implementation: CPython
Python version       : 3.14.2
IPython version      : 9.14.0

mgplot : 0.2.26
pandas : 3.0.3
readabs: 0.2.2

